# Fraus — Calibração dos três classificadores (SEM retreinar)

Este notebook **não treina modelo nenhum**. Ele ajusta um único escalar por
cabeça — a *temperatura* — sobre os checkpoints que já existem, e mede o
quanto isso aproxima a confiança do modelo da frequência empírica.

## O problema que ele resolve

Um classificador que diz `0,95` deveria acertar 95% das vezes em que diz
`0,95`. Redes neurais modernas quase nunca fazem isso: são
**sistematicamente superconfiantes**. No Fraus isso não é curiosidade
acadêmica — o score 0–100 é uma projeção direta dessas probabilidades, e a
nota e a categoria saem dele.

## Por que temperature scaling e não outra coisa

Ela divide os **logits** por um escalar `T > 0` antes do softmax. Isso muda
a confiança **sem mudar a ordem das classes** — o `argmax` é idêntico, então
acurácia, F1 e toda métrica de classificação ficam **exatamente iguais**. É a
única forma de calibração que não pode piorar a decisão do modelo.

`T > 1` amacia (o modelo era superconfiante). `T < 1` endurece.

## O que ele entrega

- `modelos/calibracao.json` — a temperatura de cada cabeça;
- **ECE antes e depois** (Expected Calibration Error);
- o **diagrama de confiabilidade** — o gráfico que fecha um slide.

## A regra que este notebook não pode violar

> **A calibração é ajustada e medida em dado que o modelo NUNCA VIU.**

Ajustar `T` no conjunto de treino daria uma temperatura que descreve o quanto
o modelo decorou, não o quanto ele erra em dado novo — e o número sairia
bonito e mentiroso. É a invariante 10 aplicada à calibração.


In [ ]:
# 1. GPU e Drive. A GPU aqui e conveniencia, nao necessidade: este notebook
#    so faz INFERENCIA (uma passada pelo conjunto cego) mais uma otimizacao
#    escalar de milissegundos. Em CPU roda, so demora mais.
!pip -q install transformers datasets scikit-learn matplotlib

import os

import torch

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', DISPOSITIVO)

from google.colab import drive
drive.mount('/content/drive')

DIR_MODELOS = '/content/drive/MyDrive/fraus/modelos'
for nome in ('bertimbau-satisfacao', 'bertimbau-emocao', 'bertimbau-ironia'):
    caminho = f'{DIR_MODELOS}/{nome}'
    assert os.path.isdir(caminho), (
        f'checkpoint ausente: {caminho}. Este notebook CALIBRA modelos ja '
        'treinados -- rode 01, 03 e 04 antes, ou copie os artefatos para o Drive.'
    )
print('os tres checkpoints estao no lugar')


## 2. O pacote `fraus`

Mesma célula dos notebooks 02, 03 e 04. A **ordem das classes** não é
redigitada aqui: vem de `fraus.sinais.*`, que é a mesma lista que o runtime
usa. Redigitar a ordem num notebook é o caminho mais curto para a invariante
8 ser violada em silêncio.


In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
DONO = 'joao-ped'
NOME_REPO = 'Fraus'
RAMO = 'main'

!rm -rf /content/fraus-src
# Clone em `fraus-src`, NUNCA em `/content/fraus`: o cwd do Colab esta no
# sys.path, e um diretorio chamado `fraus` la vira namespace package vazio
# que sequestra o import antes do pacote de verdade.
!git clone -q --branch $RAMO https://github.com/$DONO/$NOME_REPO.git /content/fraus-src
!pip -q install -e /content/fraus-src

from fraus.sinais.emocao import NOMES_EMOCOES
from fraus.sinais.texto import INSATISFEITO, NEUTRO, SATISFEITO
print('ordem de satisfacao :', INSATISFEITO, NEUTRO, SATISFEITO)
print('ordem de emocao     :', NOMES_EMOCOES)


## 3. O conjunto cego

Um por cabeça, e **cada um precisa ser dado que aquela cabeça não viu no
treino**. A forma segura é reconstruir o mesmo split, com a **mesma semente**
do notebook de treino, e ficar com a partição de teste.

> Se a semente divergir da do notebook de treino, o conjunto deixa de ser
> cego e a temperatura passa a descrever memorização. **Confira as sementes
> contra os notebooks 01, 03 e 04 antes de rodar.**

O conjunto é dividido em dois: metade **ajusta** `T`, metade **verifica** o
ECE resultante. Ajustar e reportar no mesmo dado daria um ECE otimista pelo
mesmo motivo de sempre.


In [ ]:
# 3. Conjunto cego da cabeca de SATISFACAO (B2W, mesma receita do notebook 01).
import pandas as pd
from sklearn.model_selection import train_test_split

SEMENTE = 42  # <- CONFIRA contra o notebook 01

URL_B2W = ('https://raw.githubusercontent.com/americanas-tech/'
           'b2w-reviews01/main/B2W-Reviews01.csv')
bruto = pd.read_csv(URL_B2W)

def rotular(nota):
    if nota <= 2: return 0   # insatisfeito
    if nota == 3: return 1   # neutro
    return 2                 # satisfeito

dados = bruto.dropna(subset=['review_text', 'overall_rating']).copy()
dados['rotulo'] = dados['overall_rating'].apply(rotular)

# A MESMA divisao do treino, pela MESMA semente -- e ficamos com o TESTE.
_treino, teste = train_test_split(
    dados, test_size=0.2, random_state=SEMENTE, stratify=dados['rotulo']
)
teste = teste.sample(n=min(4000, len(teste)), random_state=SEMENTE)
print('conjunto cego de satisfacao:', len(teste), 'exemplos')
print(teste['rotulo'].value_counts().sort_index().to_dict())

TEXTOS_SAT = teste['review_text'].astype(str).tolist()
ROTULOS_SAT = teste['rotulo'].tolist()


## 4. Logits — colhidos uma vez, reutilizados

A calibração trabalha sobre **logits**, não sobre probabilidades: o softmax
já destruiu a escala que a temperatura precisa ajustar. Por isso eles são
colhidos uma vez; o ajuste de `T` depois é aritmética pura e roda em
milissegundos, sem tocar no modelo de novo.


In [ ]:
# 4. Logits do conjunto cego, em lote.
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def colher_logits(caminho_modelo, textos, tamanho_maximo=192, lote=64):
    """Logits crus -- SEM softmax, que e exatamente o ponto."""
    tokenizador = AutoTokenizer.from_pretrained(caminho_modelo)
    modelo = AutoModelForSequenceClassification.from_pretrained(caminho_modelo)
    modelo.to(DISPOSITIVO).eval()
    saida = []
    with torch.inference_mode():
        for inicio in range(0, len(textos), lote):
            pedaco = textos[inicio:inicio + lote]
            entradas = tokenizador(pedaco, truncation=True,
                                   max_length=tamanho_maximo, padding=True,
                                   return_tensors='pt').to(DISPOSITIVO)
            saida.append(modelo(**entradas).logits.float().cpu().numpy())
    return np.concatenate(saida)

logits_sat = colher_logits(f'{DIR_MODELOS}/bertimbau-satisfacao', TEXTOS_SAT)
rotulos_sat = np.array(ROTULOS_SAT)
print('logits:', logits_sat.shape)


## 5. ECE e o diagrama de confiabilidade

**ECE** divide as predições em faixas pela confiança e mede, em cada faixa, a
distância entre a confiança média e a acurácia real — ponderada pelo tamanho
da faixa.

Modelo perfeitamente calibrado tem ECE = 0 e cai sobre a diagonal. Acima da
diagonal = subconfiante; **abaixo = superconfiante**, que é o caso comum e o
que dói aqui.


In [ ]:
# 5. ECE e as faixas do diagrama.
def softmax(logits, temperatura=1.0):
    z = logits / temperatura
    z = z - z.max(axis=-1, keepdims=True)   # estabilidade: exp(1000) vira inf
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def faixas_de_confiabilidade(probabilidades, rotulos, n_faixas=10):
    """Por faixa: confianca media, acuracia real e quantos exemplos caem la."""
    confianca = probabilidades.max(axis=-1)
    preditos = probabilidades.argmax(axis=-1)
    acertos = (preditos == rotulos).astype(float)
    bordas = np.linspace(0, 1, n_faixas + 1)
    linhas = []
    for baixo, alto in zip(bordas[:-1], bordas[1:]):
        dentro = (confianca > baixo) & (confianca <= alto)
        if dentro.sum() == 0:
            continue
        linhas.append({
            'faixa': f'{baixo:.1f}-{alto:.1f}',
            'n': int(dentro.sum()),
            'confianca_media': float(confianca[dentro].mean()),
            'acuracia': float(acertos[dentro].mean()),
        })
    return linhas

def ece(probabilidades, rotulos, n_faixas=10):
    linhas = faixas_de_confiabilidade(probabilidades, rotulos, n_faixas)
    total = sum(l['n'] for l in linhas)
    return sum(l['n'] / total * abs(l['confianca_media'] - l['acuracia'])
               for l in linhas)


## 6. O ajuste de `T`

Minimiza a **log-verossimilhança negativa** no split de ajuste. É um problema
de uma variável e convexo — busca ternária resolve com precisão de sobra, sem
depender de otimizador externo.

> **O `argmax` não muda.** Dividir todos os logits pelo mesmo `T > 0` preserva
> a ordem. Se a classe predita mudar depois disto, é defeito no código, não
> propriedade do método — e há uma asserção abaixo dizendo exatamente isso.


In [ ]:
# 6. Ajuste da temperatura por busca ternaria.
def nll(logits, rotulos, temperatura):
    p = softmax(logits, temperatura)
    # 1e-12 evita log(0) quando a probabilidade da classe certa satura em 0.
    return float(-np.log(p[np.arange(len(rotulos)), rotulos] + 1e-12).mean())

def ajustar_temperatura(logits, rotulos, baixo=0.05, alto=10.0, passos=200):
    for _ in range(passos):
        a = baixo + (alto - baixo) / 3
        b = alto - (alto - baixo) / 3
        if nll(logits, rotulos, a) < nll(logits, rotulos, b):
            alto = b
        else:
            baixo = a
    return (baixo + alto) / 2

def calibrar(logits, rotulos, nome, semente=SEMENTE):
    """Ajusta T em METADE do conjunto cego e reporta o ECE na outra metade."""
    aleatorio = np.random.RandomState(semente)
    ordem = aleatorio.permutation(len(rotulos))
    meio = len(ordem) // 2
    ajuste, verificacao = ordem[:meio], ordem[meio:]

    T = ajustar_temperatura(logits[ajuste], rotulos[ajuste])

    antes = softmax(logits[verificacao])
    depois = softmax(logits[verificacao], T)
    ece_antes = ece(antes, rotulos[verificacao])
    ece_depois = ece(depois, rotulos[verificacao])

    # A GUARDA DO METODO: temperature scaling NAO pode mexer na decisao.
    assert (antes.argmax(axis=-1) == depois.argmax(axis=-1)).all(), (
        'a classe predita MUDOU com a temperatura -- isso e impossivel no '
        'metodo e significa defeito no codigo acima'
    )

    melhora = (1 - ece_depois / ece_antes) * 100 if ece_antes else 0.0
    print(f'{nome}: T = {T:.4f}   ECE {ece_antes:.4f} -> {ece_depois:.4f} '
          f'({melhora:+.1f}%)')
    return {
        'temperatura': float(T),
        'ece_antes': float(ece_antes),
        'ece_depois': float(ece_depois),
        'n_ajuste': int(len(ajuste)),
        'n_verificacao': int(len(verificacao)),
        'faixas_antes': faixas_de_confiabilidade(antes, rotulos[verificacao]),
        'faixas_depois': faixas_de_confiabilidade(depois, rotulos[verificacao]),
    }

laudo = {'satisfacao': calibrar(logits_sat, rotulos_sat, 'satisfacao')}


## 7. O diagrama de confiabilidade

O gráfico que fecha o slide: as barras devem encostar na diagonal. Antes da
calibração ficam **abaixo** dela (superconfiança); depois, colam.


In [ ]:
# 7. Antes x depois, no mesmo eixo.
import matplotlib.pyplot as plt

def desenhar(nome):
    dados = laudo[nome]
    figura, eixos = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for eixo, chave, titulo in (
        (eixos[0], 'faixas_antes', f"antes  (ECE {dados['ece_antes']:.4f})"),
        (eixos[1], 'faixas_depois', f"depois (ECE {dados['ece_depois']:.4f})"),
    ):
        faixas = dados[chave]
        x = [f['confianca_media'] for f in faixas]
        y = [f['acuracia'] for f in faixas]
        eixo.plot([0, 1], [0, 1], '--', color='gray', linewidth=1,
                  label='calibração perfeita')
        eixo.bar(x, y, width=0.08, alpha=0.75)
        eixo.set_title(f'{nome} — {titulo}')
        eixo.set_xlabel('confiança predita')
        eixo.set_xlim(0, 1); eixo.set_ylim(0, 1)
    eixos[0].set_ylabel('acurácia observada')
    eixos[0].legend(loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'/content/confiabilidade_{nome}.png', dpi=150)
    plt.show()

desenhar('satisfacao')


## 8. As outras duas cabeças

**Emoção** usa o split cego do notebook 03 (GoEmotions PT-BR reduzido a
Ekman). **Ironia**, o do notebook 04.

> A cabeça de ironia tem limitação declarada: erra 6 em 10 falas sinceras de
> atendimento com 0,999 de confiança, e **por isso não pontua** desde
> 04/09/2026. Calibrá-la **não conserta isso** — temperatura ajusta o quanto
> o modelo se diz confiante, não *sobre o que* ele está confiante. O ECE dela
> vai melhorar e ela vai continuar errando o domínio. Apresentar a melhora de
> ECE como conserto seria a desonestidade que este projeto recusa.


In [ ]:
# 8. Emocao e ironia -- monte os conjuntos cegos com a receita dos notebooks
#    03 e 04, e as MESMAS sementes deles.
#
# TEXTOS_EMO, ROTULOS_EMO = ...   (GoEmotions PT-BR, split de teste do 03)
# TEXTOS_IRO, ROTULOS_IRO = ...   (corpus do 04, split de teste)
#
# logits_emo = colher_logits(f'{DIR_MODELOS}/bertimbau-emocao', TEXTOS_EMO)
# laudo['emocao'] = calibrar(logits_emo, np.array(ROTULOS_EMO), 'emocao')
# desenhar('emocao')
#
# logits_iro = colher_logits(f'{DIR_MODELOS}/bertimbau-ironia', TEXTOS_IRO)
# laudo['ironia'] = calibrar(logits_iro, np.array(ROTULOS_IRO), 'ironia')
# desenhar('ironia')
print('cabecas calibradas:', list(laudo))


## 9. Export

`calibracao.json` vai para a **raiz** de `modelos/`, ao lado de
`importancias.json` — não para dentro de um diretório de checkpoint. O
motivo: ele descreve uma relação entre a cabeça e um conjunto cego, e não é
parte do artefato do modelo.

> **Nada no runtime consome este arquivo ainda, e isso é de propósito.**
> Aplicar a temperatura muda o score de toda conversa do banco — e a régua
> deste projeto para trocar o que gera número é medir o efeito em
> `score`/`nota`/`categoria` **antes**, não depois. A mesma disciplina que
> recusou a quantização int8 (`docs/encolhimento.md`) vale aqui.


In [ ]:
# 9. Export do laudo.
import json

CAMINHO = f'{DIR_MODELOS}/calibracao.json'
with open(CAMINHO, 'w', encoding='utf-8') as arquivo:
    json.dump(laudo, arquivo, ensure_ascii=False, indent=2)
print('gravado em', CAMINHO)

for nome, dados in laudo.items():
    print(f"  {nome:12} T={dados['temperatura']:.4f}  "
          f"ECE {dados['ece_antes']:.4f} -> {dados['ece_depois']:.4f}")


## Para onde levar

```
modelos/calibracao.json          <- este notebook
confiabilidade_satisfacao.png    <- o slide
```

E o que **não** fazer: aplicar a temperatura no runtime sem antes medir o
deslocamento de `score`, `nota` e `categoria` sobre um conjunto de conversas.
`scripts/comparar_backends.py` já faz exatamente essa medição para outra
mudança, e serve de molde.
